# Validation — `kappa-lora-halve-params`

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`4a4fc805f17f`](https://github.com/mayorquinmachines/peft/commit/4a4fc805f17f398d6c47b9577b3a3696f6f54868)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/kappa-lora/llama-3.2-3B-rank32` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Verdict: `unknown`** · Remyx run `ab259cd5`

The outputs below are the ones that run produced on Remyx compute. Run the cells top to bottom on a machine with a GPU to reproduce them.

> **This run did not finish.** preflight: experiments: the harness is given `experiments/kappa-lora/llama-3.2-3B-rank32`, which does not exist at this commit — sibling configs live beside the `kappa-lora` ones. The run would invoke the harness with a path it cannot read and produce no results.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "4a4fc805f17f398d6c47b9577b3a3696f6f54868"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/kappa-lora/llama-3.2-3B-rank32` (relative to `method_comparison/MetaMathQA`).

In [ ]:
import glob
for path in sorted(glob.glob(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/kappa-lora/llama-3.2-3B-rank32/*"))):
    print(f"--- {path} ---")
    print(open(path).read())

## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/kappa-lora/llama-3.2-3B-rank32` — a directory of experiments runs each in turn; a single experiment runs once.

In [ ]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys
configs = sorted(glob.glob("experiments/kappa-lora/llama-3.2-3B-rank32/*/")) or ["experiments/kappa-lora/llama-3.2-3B-rank32"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

(no log captured)

## 7. Read what the benchmark wrote

Results land under `results/kappa-lora--*.json` (relative to `method_comparison/MetaMathQA`). The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["results/kappa-lora--*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test_accuracy"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

(the benchmark wrote no result document — see the run above)


## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 29176627,
        "baseline": 48627712
    },
    {
        "metric": "test_accuracy",
        "direction": ">=",
        "threshold": 0.4,
        "baseline": null
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    print(f"{c['metric']:<28}{str(v):>16}{str(c['baseline']):>16}  {c['direction']} {t}  {mark}")

## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

## 10. What the outcome means

**Verdict: `unknown`.** The run did not produce a measurement: preflight: experiments: the harness is given `experiments/kappa-lora/llama-3.2-3B-rank32`, which does not exist at this commit — sibling configs live beside the `kappa-lora` ones. The run would invoke the harness with a path it cannot read and produce no results. Every change the validation loop made on the way is a separate commit on this branch, named for the run that applied it — revert any of them to undo it.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
benchmarks:
  - name: kappa-lora-halve-params
    suite:
      harness:
        notebook: .remyx/validations/kappa-lora-halve-params.ipynb
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/kappa-lora/llama-3.2-3B-rank32
        results_glob: method_comparison/MetaMathQA/results/kappa-lora--*.json
        method: kappa-lora
    scorer: num_trainable_params
    # the published standard-LoRA row IS the baseline, so only the feature arm runs;
    # the kappa-lora config enables the default-off flag condition_number_top_fraction=0.5
    baseline:
      source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
      values:
        # analytic standard-LoRA count for r=32 over q/k/v/o/gate/up/down of Llama-3.2-3B (28 layers):
        # 28 * (2*196608 + 2*131072 + 3*360448) = 48,627,712
        num_trainable_params: 48627712
    metrics:
      - name: num_trainable_params
        direction: min
        # 0.60 x 48,627,712: top half of the 196 matched modules by condition number is expected near 24.3M;
        # allows skew to ~0.6x if the larger MLP projections dominate the ranking while still demanding a >=40% cut
        threshold: 29176627
        role: target
      - name: test_accuracy
        direction: max
        # in-distribution MetaMathQA eval of a rank-32-tuned Llama-3.2-3B scores well above 0.4;
        # this floor (cleared by the published LoRA row) flags a >10-point fit collapse
        threshold: 0.4
        role: guardrail
    policy:
      guardrail_veto: true
    compute:
      tier: gpu
      # one arm: ~7 GB gated model download + LoRA SFT on the harness MetaMathQA subset (~1k steps x ~3 s/step on one A100) + eval ~= 2 h; 4 h leaves headroom
      timeout_s: 14400
    held_constant:
      - "base model meta-llama/Llama-3.2-3B"
      - "the harness's default_training_params.json (learning rate, epochs, batch size, seed)"
      - "LoRA rank r=32 and the seven target projections, matching the published lora--llama-3.2-3B-rank32 row"
      - "identical eval split and scoring as the published corpus"
    avoid:
      - "editing training_params.json or any harness default (breaks comparability with the published corpus)"
      - "unpinned base-model or dataset revisions drifting between runs"
      - "wall-clock or throughput comparisons (not the mechanism under test)"
    provenance:
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      num_trainable_params: "user_guidance"
      test_accuracy: "maintainer_comment:@mayorquinmachines"
      baseline: "published corpus method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      baseline_num_trainable_params: "inferred"
      num_trainable_params_threshold: "inferred"
      test_accuracy_threshold: "inferred"
      held_constant: "protocol_doc:method_comparison/README.md"
      experiments: "synthesized"
```